<a href="https://colab.research.google.com/github/NazmulHudaNabil/email-spam-detection/blob/main/Spam_or_ham_using_machine_learning_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df = pd.read_csv("/content/spam.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
df.shape

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
df['Category'] = encoder.fit_transform(df['Category'])

In [ ]:
df["Category"]

### Process Text

In [ ]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')


stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

### Text Clean

In [ ]:
import string
def process_text(text):
  # Lower Case
  text = text.lower()

  # Remove punctuation
  text = ''.join([char for char in text if char not in string.punctuation])

  # Remove Stopwords
  words = text.split()
  words = [word for word in words if word not in stop_words]

  # Lematization
  words = [lemmatizer.lemmatize(word) for word in words]

  return ' '.join(words)

In [ ]:
process_text("Hello, I am studying, and she is reading; however, we are not learning. The children are playing in the garden.")

In [ ]:
df["Message"] = df["Message"].apply(process_text)

In [ ]:
df["Message"][0]

### Tf-IDF feature Extraction

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,     # limit vocab size
    ngram_range=(1,2),     # unigram + bigram
    min_df=2,              # ignore rare words
    max_df=0.8             # ignore very common words
)

X = vectorizer.fit_transform(df["Message"])

print(X.shape)
print(X[0])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, df["Category"], test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
y_train[1]

### Multiple Model to check recall score

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, recall_score

navie_bayes = MultinomialNB()
random_forest = RandomForestClassifier()

models = {
    'navie_bayes':navie_bayes,
    'random_forest':random_forest,
    'logistic_regression':LogisticRegression(),
    'svm':SVC(),
    "xgboost":XGBClassifier()
}


for name, model in models.items():
  model.fit(X_train, y_train)
  print(f"{name}, Training : {model.score(X_test, y_test)}")

  y_pred = model.predict(X_test)
  print(f"{name} recall Score: {recall_score(y_test, y_pred)}")
  print()

In [ ]:
!pip install optuna

### Model tuning

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def objective(trial):
    model_name = trial.suggest_categorical(
        'model', ['naive_bayes', 'random_forest', 'xgboost']
    )

    if model_name == 'naive_bayes':
        model = MultinomialNB(
            alpha=trial.suggest_float('nb_alpha', 1e-3, 1e3, log=True),
            fit_prior=trial.suggest_categorical('nb_fit_prior', [True, False]),
        )

    elif model_name == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int('rf_n_estimators', 100, 500),
            max_depth=trial.suggest_int('rf_max_depth', 2, 20),
            min_samples_split=trial.suggest_int('rf_min_samples_split', 2, 10),
            min_samples_leaf=trial.suggest_int('rf_min_samples_leaf', 1, 4),
            max_features=trial.suggest_categorical('rf_max_features', ['sqrt', 'log2', None]),
            random_state=42,
            n_jobs=-1
        )

    elif model_name == 'xgboost':
        model = XGBClassifier(
            n_estimators=trial.suggest_int('xgb_n_estimators', 100, 500),
            learning_rate=trial.suggest_float('xgb_learning_rate', 1e-3, 1.0, log=True),
            max_depth=trial.suggest_int('xgb_max_depth', 3, 10),
            subsample=trial.suggest_float('xgb_subsample', 0.5, 1.0),
            colsample_bytree=trial.suggest_float('xgb_colsample_bytree', 0.5, 1.0),
            reg_alpha=trial.suggest_float('xgb_reg_alpha', 1e-5, 1e2, log=True),
            reg_lambda=trial.suggest_float('xgb_reg_lambda', 1e-5, 1e2, log=True),
            gamma=trial.suggest_float('xgb_gamma', 1e-5, 1e2, log=True),
            min_child_weight=trial.suggest_int('xgb_min_child_weight', 1, 10),
            random_state=42,
            eval_metric='logloss',
            n_jobs=-1
        )

    score = cross_val_score(
        model,
        X,
        df["Category"],
        cv=5,
        scoring='recall',
        n_jobs=-1
    ).mean()

    return score


optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)  # increased trials

print(f"Best Recall Score: {study.best_value:.4f}")
print(f"Best Parameters: {study.best_params}")

In [ ]:
# Best Recall Score: 0.9438
# Best Parameters: {'model': 'naive_bayes', 'nb_alpha': 0.1327555116}